## 2、工具的定义方式1：不使用 @tool

对应《尚硅谷-05-Tools.pdf》第 2 节（PDF 第 7–14 页）。

重点：普通 Python 函数也能传给 `bind_tools()`。
先离线查看工具描述，最后再按需运行真实模型示例。
本文件可独立学习，不依赖其他 Notebook 的变量；选择 `langchain` 内核。

### 2.1 普通函数也可以作为工具

保留函数名、类型注解和文档字符串，LangChain 会据此生成工具描述。
普通函数仍通过 `get_weather(...)` 调用，没有工具对象的 `.invoke()`。

In [1]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city: str) -> str:
    return f"{city}天气晴朗~"  # 模拟数据

print(get_weather("北京"))
rprint(convert_to_openai_tool(get_weather))

北京天气晴朗~


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 2.2 工具描述的各部分

#### 2.2.1 convert_to_openai_tool：查看发送给模型的描述

在本笔记沿用的 OpenAI 兼容模型中，可以用这个函数观察工具 Schema。
它只转换描述，**不会执行 get_weather**。

| 字段 | 从哪里来／有什么用 |
| --- | --- |
| 外层 `type="function"` | 表示这是函数工具 |
| `function.name` | 默认来自 Python 函数名 |
| `function.description` | 主要来自 docstring |
| `parameters.type="object"` | 参数是一组键值对 |
| `properties` | 每个参数的名称、类型、说明、默认值 |
| `required` | 没有默认值的必填参数 |

参数中的 `type` 是 JSON Schema 类型，例如 `string`、`integer`、`number`、`boolean`、`array`、`object`。

In [2]:
tool_schema = convert_to_openai_tool(get_weather)
function_schema = tool_schema["function"]

print("工具名：", function_schema["name"])
print("描述：", repr(function_schema["description"]))
rprint(function_schema["parameters"])

工具名： get_weather
描述： ''


{'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}

#### 2.2.2 description：来自 docstring

上例没有 docstring，所以工具描述为空。补上文档字符串，再观察变化。

In [3]:
def get_weather(city: str) -> str:
    """天气查询工具。"""
    return f"{city}天气晴朗~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具。',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

#### 2.2.3 参数说明：使用 Google 风格 docstring

摘要后空一行，再写 `Args:`；参数名与函数签名保持一致。
普通函数转换时会解析参数说明，后面 `@tool` 的默认行为与它不同。

In [4]:
def get_weather(city: str) -> str:
    """天气查询工具。

    Args:
        city: 城市名称，例如北京、上海。
    """
    return f"{city}天气晴朗~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具。',
        'parameters': {
            'properties': {'city': {'description': '城市名称，例如北京、上海。', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

#### 2.2.4 参数类型：来自类型注解

`city: str` 对应 JSON Schema 的 `"type": "string"`。
去掉类型注解，并且不写 `Args:` 时，参数仍存在，但缺少类型信息。

In [5]:
def get_weather(city):
    """天气查询工具。"""
    return f"{city}天气晴朗~"

rprint(convert_to_openai_tool(get_weather)["function"]["parameters"])

{'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}

##### 易错：Args 中写了参数，却没有对应的类型注解

下面是课件里的报错示例，捕获并打印预期异常，方便继续执行后续单元。

In [6]:
def get_weather(city):
    """天气查询工具。

    Args:
        city: 城市名称。
    """
    return f"{city}天气晴朗~"

try:
    convert_to_openai_tool(get_weather)
except ValueError as e:
    print(type(e).__name__, e)

ValueError Arg city in docstring not found in function signature.


#### 2.2.5 参数默认值：决定 required 和 default

只有一个带默认值的参数时，不需要 `required` 字段；默认值会出现在参数描述中。

In [7]:
def get_weather(city: str = "北京") -> str:
    """天气查询工具。

    Args:
        city: 城市名称。
    """
    return f"{city}天气晴朗~"

rprint(convert_to_openai_tool(get_weather)["function"]["parameters"])
print(get_weather())

{'properties': {'city': {'default': '北京', 'description': '城市名称。', 'type': 'string'}}, 'type': 'object'}

北京天气晴朗~


##### 再加一个没有默认值的 dt 参数

此时 `dt` 必填，`city` 可省略。`dt` 在这里表示日期。

In [8]:
def get_weather(dt: str, city: str = "北京") -> str:
    """天气查询工具。

    Args:
        dt: 日期。
        city: 城市名称。
    """
    return f"{dt}，{city}天气晴朗~"  # 仍是模拟数据

rprint(convert_to_openai_tool(get_weather)["function"]["parameters"])
print(get_weather(dt="2026-10-16"))

{
    'properties': {
        'dt': {'description': '日期。', 'type': 'string'},
        'city': {'default': '北京', 'description': '城市名称。', 'type': 'string'}
    },
    'required': ['dt'],
    'type': 'object'
}

2026-10-16，北京天气晴朗~


### 2.3 绑定模型并发送请求（需要 API）

这部分沿用你的百炼配置，可按需执行。重新定义只含 `city` 的示例，避免受上面的 `dt` 参数影响。
模型名沿用原笔记，真实接口未在本次整理中请求。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 沿用之前笔记的百炼配置；初始化本身不会发送模型请求
load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=BAILIAN_API_KEY,
    base_url=BAILIAN_API_URL,
)

In [ ]:
def get_weather(city: str) -> str:
    """天气查询工具。

    Args:
        city: 城市名称。
    """
    return f"{city}天气晴朗~"

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("今天北京天气如何？")
rprint(response)
print(response.tool_calls)

### 2.4 小结

`函数名 → name`；`docstring → description`；`类型注解 → type`；`默认值 → default / required`。

普通函数可以绑定到模型，但模型只返回调用请求。
执行时仍需 `get_weather(**tool_call["args"])`，再手动构造 `ToolMessage`（见第一份笔记）。

下一节使用 `@tool`，把函数封装成可以 `.invoke()` 的工具对象。

参考：[工具定义](https://docs.langchain.com/oss/python/langchain/tools)、
[Google 风格 docstring 示例](https://www.sphinx-doc.org/en/master/usage/extensions/example_google.html)。